In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [ ]:
import string

punctuations = list(string.punctuation) # Get all punctuation : !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~

In [ ]:
import kagglehub
import pandas as pd
import os


path = kagglehub.dataset_download("emineyetm/fake-news-detection-datasets")


data_path = os.path.join(path, "News _dataset")

print("Files:", os.listdir(data_path))


fake_df = pd.read_csv(os.path.join(data_path, "Fake.csv"))
true_df = pd.read_csv(os.path.join(data_path, "True.csv"))

print(fake_df.head())
print(true_df.head())

Using Colab cache for faster access to the 'fake-news-detection-datasets' dataset.
Files: ['True.csv', 'Fake.csv']
                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas day, Donald Trump announced that ...    News   
4  Pope Francis used his annual Christmas Day mes...    News   

                date  
0  December 31, 2017  
1  December 31, 2017  
2  December 30, 2017  
3  December 29, 2017  
4  December 25, 2017  
                         

In [ ]:
fake_df.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [ ]:
true_df.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [ ]:
fake_df['label'] = 0
true_df['label'] = 1

In [ ]:
true_df_cols = true_df.columns
fake_df_cols = fake_df.columns

print (true_df_cols)
print (fake_df_cols)

Index(['title', 'text', 'subject', 'date', 'label'], dtype='object')
Index(['title', 'text', 'subject', 'date', 'label'], dtype='object')


In [ ]:
data = pd.concat([fake_df, true_df], ignore_index=True)
data = data.sample(frac=1, random_state=42).reset_index(drop=True) #shuffle the dataset

data.head()

,title,text,subject,date,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",0
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",1
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",1
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",0
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",1


In [ ]:
X = data['text'].tolist()[:500]
y = data['label'].tolist()[:500]
print(X)

['21st Century Wire says Ben Stein, reputable professor from, Pepperdine University (also of some Hollywood fame appearing in TV shows and films such as Ferris Bueller s Day Off) made some provocative statements on Judge Jeanine Pirro s show recently. While discussing the halt that was imposed on President Trump s Executive Order on travel. Stein referred to the judgement by the 9th Circuit Court in Washington state as a  Coup d tat against the executive branch and against the constitution.  Stein went on to call the Judges in Seattle  political puppets  and the judiciary  political pawns. Watch the interview below for the complete statements and note the stark contrast to the rhetoric of the leftist media and pundits who neglect to note that no court has ever blocked any Presidential orders in immigration in the past or discuss the legal efficacy of the halt or the actual text of the Executive Order.READ MORE TRUMP NEWS AT: 21st Century Wire Trump FilesSUPPORT OUR WORK BY SUBSCRIBING 

In [ ]:
!pip install num2words > /dev/null

In [ ]:
from num2words import num2words
import re

def convert_digits(sentence):
    # expression régulière pour trouver les nombres et les ordinaux (1st, 2nd, etc.)
    pattern = r'\d+(st|nd|rd|th)?'

    # fonction pour remplacer chaque nmbr trouvé
    def change_match(m):
        value = m.group()

        # vérifier nmbr est un ordinal
        check_ord = re.match(r'(\d+)(st|nd|rd|th)', value)
        if check_ord:
            n = int(check_ord.group(1))
            word = num2words(n, to="ordinal")
        else:
            n = int(value)
            word= num2words(n)
        return word

    # remplacer tous les nombres dans phrase
    updated_sentence = re.sub(pattern, change_match, sentence)
    return updated_sentence

In [ ]:
numbers_vocab = {
    "zero","one","two","three","four","five","six","seven","eight","nine",
    "ten","eleven","twelve","thirteen","fourteen","fifteen","sixteen",
    "seventeen","eighteen","nineteen","twenty","thirty","forty","fifty",
    "sixty","seventy","eighty","ninety","hundred","thousand",
    "first","second","third","fourth","fifth","sixth","seventh","eighth","ninth","tenth",
}

def drop_stopwords(phrase):

    # tokenisation avec spacy
    doc = nlp(phrase)

    tokens = list(doc)
    result = []
    i = 0
    while i < len(tokens):
        tok = tokens[i]
        # garder le mot si n'est pas stop word ou si c'est un mot de nmbr
        if (not tok.is_stop) or (tok.text.lower() in numbers_vocab):

            # lemmatisation + minuscule
            result.append(tok.lemma_.lower())
        i += 1
    return " ".join(result)

In [ ]:
!pip install contractions

In [ ]:
def dropunctuation(phrase):
    #tokenisation avec spacy
    doc = nlp(phrase)
    tokens = list(doc)
    cleaned_tok = []

    i = 0
    while i < len(tokens):
        tok = tokens[i]

        # ignorer la ponctuation et les espaces
        if (not tok.is_punct) and (not tok.is_space):
            cleaned_tok.append(tok)
        i += 1
    return cleaned_tok

In [ ]:
processed_texts = []

i = 0
while i < len(X):

    phrase = X[i]
    phrase = phrase.lower()
    phrase = convert_digits(phrase)
    phrase = drop_stopwords(phrase)
    phrase = dropunctuation(phrase)
    processed_texts.append(phrase)

    i += 1
print(processed_texts)
print(type(processed_texts[0]))
print(type(processed_texts))


[[twenty, first, century, wire, say, ben, stein, reputable, professor, pepperdine, university, hollywood, fame, appear, tv, show, film, ferris, bueller, s, day, provocative, statement, judge, jeanine, pirro, s, recently, discuss, halt, impose, president, trump, s, executive, order, travel, stein, refer, judgement, ninth, circuit, court, washington, state, coup, d, tat, executive, branch, constitution, stein, go, judge, seattle, political, puppet, judiciary, political, pawn, watch, interview, complete, statement, note, stark, contrast, rhetoric, leftist, medium, pundit, neglect, note, court, block, presidential, order, immigration, past, discuss, legal, efficacy, halt, actual, text, executive, order.read, trump, news, twenty, first, century, wire, trump, filessupport, work, subscribe, member, @twenty, onewire.tv], [washington, reuters, u.s, president, donald, trump, remove, chief, strategist, steve, bannon, national, security, council, wednesday, reverse, controversial, decision, early,

In [ ]:
# ensemble pour stocker les mots uniques
vocab_set = set()
for phrases in processed_texts:
    tokens = list(phrases)
    for tok in tokens:
        # minuscule pour eviter doublons
        vocab_set.add(tok.text.lower())

# trier le vocabulaire
vocab_list = sorted(vocab_set)
# construire le vocabulaire apres le traitement du texte
print("Vocabulary:", vocab_list)
print(len(vocab_list))


Vocabulary: ['$', '+', '-andrew', '-independentthe', '-nineteen', '-one', '-or-', '-robert', '.@realdonaldtrump', '.@rkadyrov', '.brat', '.cnn', '.dure', '.forty', '.here', '.how', '.https://youtu.be/gbjpeightzxduwk', '.i', '.in', '.more', '.now', '.source', '.spxrt', '.the', '.viewer', '.watch', '.with', '//connect.facebook.net', '<', '=', '@aaldef', '@abc', '@abcpolitics', '@advodude', '@alpunto', '@alxthomp', '@andrewchamings', '@angelojohngage', '@antoniodelotero', '@babyboogaloo', '@barbmuenchen', '@beyonce', '@biancaamarilis', '@billmaher', '@billstl', '@blisstabitha', '@blob_fish', '@boycotkochbro', '@braddjaffy', '@brfreed', '@bttrcrmbakeshop', '@buzzfeedandrew', '@carlbernstein', '@carminezozzora', '@cbs', '@cbsnews', '@cernovich', '@chriscuomo', '@cnn', '@cnnpolitics', '@collinrugg', '@comebackteamtwelve', '@cooksphere', '@corrynmb', '@culturalcombat', '@dan_gartland', '@dankbeans', '@davidgmcafee', '@davidmackau', '@davisonvideo', '@diamondandsilk', '@disneymommytwenty', '@d

In [ ]:
bow_vectors = []
for phrase in processed_texts:
    vec = [0] * len(vocab_list)

    for tok in phrase:
        if tok.text.lower() in vocab_list:
            index = vocab_list.index(tok.text.lower())
            vec[index] = 1

    bow_vectors.append(vec)

In [ ]:
import numpy as np

X_vectors = np.array(bow_vectors) #define your Vectors
y_array = np.array(y)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_vectors, y_array, test_size=0.2, random_state=42, stratify=y_array
)

In [ ]:
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score, classification_report

nb = BernoulliNB()
nb.fit(X_train, y_train)    # Train

y_pred = nb.predict(X_test) # Predict



In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.91
              precision    recall  f1-score   support

           0       1.00      0.81      0.89        47
           1       0.85      1.00      0.92        53

    accuracy                           0.91       100
   macro avg       0.93      0.90      0.91       100
weighted avg       0.92      0.91      0.91       100

